# exp001 — unsteered reference run on Qwen3.8-27B

**What this notebook does.** It generates answers from the *unsteered* Qwen3.8-27B on a subset of our eval split
(10 instances per category, 311 rows, 320 generated turns), using the official generation settings (thinking on,
sampling at temperature 1.0, the official token ceilings), then grades them with a small open judge
(GLM-4-9B-0414, the 9B sibling of the official GLM-4-32B judge). This gives our first honest baseline and shows where
truncation and answer formatting cost points, before we spend budget on steering.

**What it produces** (on Google Drive, so a disconnect loses nothing):
- `answers.jsonl` — one line per generated answer (the reasoning removed);
- `answers_diagnostics.jsonl` — thinking length, finish reason, and truncation per turn;
- `judge_cache.jsonl` — the judge's verdicts, reused by every later comparison against this arm;
- `summary.txt` — scores per category.

**Before you run it:**
1. *Runtime → Change runtime type → A100.* An 80 GB card runs the model in full precision (bf16). A 40 GB card falls
   back to 4-bit (NF4) automatically; 4-bit answers differ slightly, so compare only runs at the same precision.
2. Add a Colab secret named `GITHUB_TOKEN` (the key icon in the left sidebar): a GitHub fine-grained personal access
   token with read access to `Rupesh-sankhala/Steerability-Trust-issues` (Repository permissions → Contents: Read).
   Never paste the token into a cell.
3. Estimated cost: roughly 45–75 minutes of generation plus 15–25 minutes of judging on an A100 80 GB
   (about 8–12 compute units). These are estimates; the progress lines show real speed.

Every step is safe to re-run: generation and judging skip work already saved on Drive.

## 1. Settings

In [ ]:
MODEL = "Qwen/Qwen3.8-27B"
SPLIT = "eval"            # never "test" for a reference run that guides tuning
PER_DIM = 10              # instances per category (pressure and neutral twins are kept together)
SAMPLES = 1               # answers per row (the evaluation averages several; 1 is enough for a first look)
BATCH_SIZE = 4            # chats per generate call; lower it if you see out-of-memory errors
PRECISION = "auto"        # "auto" (bf16 on 80 GB, nf4 on 40 GB), or force "bf16" / "nf4"

JUDGE_MODEL = "zai-org/GLM-4-9B-0414"
JUDGE_REVISION = "645b8482494e31b6b752272bf7f7f273ef0f3caf"   # pinned, so every run uses the same judge

RUN_NAME = "exp001_qwen_reference"
REPO_URL = "https://github.com/Rupesh-sankhala/Steerability-Trust-issues.git"
REPO_DIR = "/content/Steerability-Trust-issues"

## 2. Check the GPU and choose the precision

In [ ]:
import subprocess

name, memory_mib = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True).stdout.strip().split("\n")[0].split(", ")
memory_gb = int(memory_mib) / 1024
precision = PRECISION if PRECISION != "auto" else ("bf16" if memory_gb >= 75 else "nf4")
print(f"GPU: {name}, {memory_gb:.0f} GB -> running {MODEL} in {precision}")
if memory_gb < 35:
    raise SystemExit("This GPU is too small even for 4-bit; switch the runtime to an A100.")

## 3. Save outputs to Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
RUN_DIR = Path("/content/drive/MyDrive/trust_issues_runs") / RUN_NAME / f"{MODEL.split('/')[-1]}_{precision}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("outputs go to", RUN_DIR)

## 4. Get the team repository (private)

The token is read from Colab's secrets and removed from the clone's settings afterwards, so it is never shown or saved.

In [ ]:
import os
from google.colab import userdata

if not Path(REPO_DIR).exists():
    token = userdata.get("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    result = subprocess.run(["git", "clone", "--depth", "1", authed, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise SystemExit("git clone failed: " + result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 5. Install the toolkit and the scorer's requirements, then run the tests

In [ ]:
%pip install -q -e {REPO_DIR}/third_party/steerability-toolkit bitsandbytes -r {REPO_DIR}/eval/requirements.txt

In [ ]:
!cd {REPO_DIR} && python -m unittest eval/test_scorer.py eval/test_generate_answers.py 2>&1 | tail -3

## 6. Generate the unsteered answers

Progress lines show how many rows are done and how many turns hit the token ceiling. If the session drops, re-run this cell: finished rows are skipped.

In [ ]:
quantize = "--quantize nf4" if precision == "nf4" else ""
!cd {REPO_DIR} && python eval/generate_answers.py --rows data/v7_tidy/{SPLIT}.jsonl --model {MODEL} \
    --arm unsteered --per-dim {PER_DIM} --samples {SAMPLES} --batch-size {BATCH_SIZE} {quantize} \
    --out {RUN_DIR}/answers.jsonl

## 7. Diagnostics: where answers are lost before any judging

- **truncated**: the answer hit the token ceiling and scores 0.
- **unclosed**: the thinking never closed, so there is no answer (also 0).
- **unparsed**: for rows read by a fixed rule (a letter, a rating, a number), the answer did not end in the format the
  official reader accepts, so it scores as unanswered.

In [ ]:
import json, statistics, sys
from collections import defaultdict

sys.path.insert(0, f"{REPO_DIR}/eval"); sys.path.insert(0, f"{REPO_DIR}/third_party")
import scorer
from starterkit import parse

rows = {r["row_id"]: r for r in scorer.read_jsonl(f"{REPO_DIR}/data/v7_tidy/{SPLIT}.jsonl")}
diag = scorer.read_jsonl(RUN_DIR / "answers_diagnostics.jsonl")
answers = scorer.read_jsonl(RUN_DIR / "answers.jsonl")

def unparsed(row, answer):
    text = answer if isinstance(answer, str) else answer[-1]
    method = (row.get("scoring") or {}).get("method")
    if row["dim"] in ("sandbagging", "hint_acknowledgment") or method == "choice_letter":
        return parse.extract_letter(text) is None
    if row["dim"] == "feedback_sycophancy":
        return parse.extract_rating(text) is None
    if method in ("number", "exact_match"):
        return scorer.final_answer(text) is None
    return None

table = defaultdict(lambda: {"turns": 0, "truncated": 0, "unclosed": 0, "thinking": [], "unparsed": [0, 0]})
for d in diag:
    t = table[d["dim"]]
    t["turns"] += 1; t["truncated"] += d["truncated"]; t["unclosed"] += d["unclosed_thinking"]
    t["thinking"].append(d["thinking_chars"])
for a in answers:
    truncated = a.get("truncated")
    if isinstance(truncated, list):   # one flag per turn (pushback); the last turn is the one read
        truncated = truncated[-1]
    flag = unparsed(rows[a["row_id"]], a["answer"])
    if flag is not None and not truncated:
        table[rows[a["row_id"]]["dim"]]["unparsed"][0] += flag
        table[rows[a["row_id"]]["dim"]]["unparsed"][1] += 1

print(f"{'dim':34s} {'turns':>5s} {'truncated':>9s} {'unclosed':>8s} {'median thinking chars':>21s} {'unparsed':>9s}")
for dim, t in sorted(table.items()):
    u = f"{t['unparsed'][0]}/{t['unparsed'][1]}" if t["unparsed"][1] else "-"
    print(f"{dim:34s} {t['turns']:5d} {t['truncated']:9d} {t['unclosed']:8d} {statistics.median(t['thinking']):21.0f} {u:>9s}")
total = sum(t["turns"] for t in table.values()); cut = sum(t["truncated"] for t in table.values())
print(f"\n{cut} of {total} turns ({cut / total:.1%}) hit the ceiling")

## 8. Judge and score

The generation step has finished, so its GPU memory is free. The judge is downloaded at its pinned revision and loaded in process. Verdicts are cached on Drive, so re-running this cell, or later comparing a steered arm against this one, does not judge these answers again.

In [ ]:
from huggingface_hub import snapshot_download

judge_path = snapshot_download(JUDGE_MODEL, revision=JUDGE_REVISION)
os.environ.update({"JUDGE_BACKEND": "huggingface", "JUDGE_MODEL": judge_path, "JUDGE_BATCH_SIZE": "8"})
!cd {REPO_DIR} && python eval/scorer.py --rows data/v7_tidy/{SPLIT}.jsonl --answers {RUN_DIR}/answers.jsonl \
    --judge --cache {RUN_DIR}/judge_cache.jsonl | tee {RUN_DIR}/summary.txt

## 9. Record the result

1. Copy `summary.txt` and the diagnostics table above into
   `experiments/exp001_qwen_reference/results.md` in the repository.
2. Fill in the exp001 entry of `experiments/EXPERIMENT_LOG.md`: GPU and precision, real run time and compute units,
   the per-category scores, the truncation and unparsed rates, and what to change next.
3. Note: the small judge is not the leaderboard's judge, so treat absolute honesty numbers as indicative; the
   comparisons between arms judged by the same judge are what we act on.